# CAARMA nationality-conditioned mixup on Bridges-2

Run this notebook from the repository root in a Bridges-2 Open OnDemand Jupyter session with a GPU allocation. Keep datasets and checkpoints in `$PROJECT`; do not train on a login node. The batch alternative is `sbatch -A <allocation> bridges2/train_nationality.sbatch`.

Before opening Jupyter, inspect available PSC environments with `module spider AI`. Load the selected AI/PyTorch module, create `$HOME/.venvs/caarma`, and install `requirements_bridges2.txt`. Set `AI_MODULE` to that exact module name when using the batch script.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import socket
import subprocess
import sys

import pandas as pd
import torch
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'mixup_nationality.py').exists():
    raise RuntimeError('Run from the CAARMA repo root.')
PROJECT = Path(os.environ['PROJECT']).resolve()
DATA_ROOT = Path(os.environ.get('CAARMA_DATA_ROOT', PROJECT / 'caarma-data')).resolve()
TRAIN_ROOT = Path(os.environ.get('CAARMA_TRAIN_ROOT', DATA_ROOT / 'voxceleb1/vox1_dev_wav/wav')).resolve()
TEST_ROOT = Path(os.environ.get('CAARMA_TEST_ROOT', DATA_ROOT / 'voxceleb1_test/wav')).resolve()
TRAIN_CSV = Path(os.environ.get('CAARMA_TRAIN_CSV', DATA_ROOT / 'voxceleb_full.csv')).resolve()
TRIAL_PATH = Path(os.environ.get('CAARMA_TRIAL_PATH', PROJECT_ROOT / 'data/vox1_test.txt')).resolve()
META_PATH = Path(os.environ.get('CAARMA_META_PATH', PROJECT_ROOT / 'vox1_meta.csv')).resolve()
RUN_ID = os.environ.get('SLURM_JOB_ID') or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
SAVE_DIR = Path(os.environ.get('CAARMA_SAVE_DIR', PROJECT / 'caarma-output/nationality')).resolve() / RUN_ID
CONFIG_PATH = SAVE_DIR / 'config_nationality_bridges2.generated.yaml'
NUM_WORKERS = int(os.environ.get('SLURM_CPUS_PER_TASK', '5'))

def require_project_output(path):
    path = path.resolve()
    if not path.is_relative_to(PROJECT):
        raise ValueError(f'Output must remain under PROJECT ({PROJECT}): {path}')
    return path

TRAIN_CSV = require_project_output(TRAIN_CSV)
SAVE_DIR = require_project_output(SAVE_DIR)

print('Host:', socket.gethostname())
print('Python:', sys.executable)
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU visible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
if not torch.cuda.is_available():
    raise RuntimeError('Request a Bridges-2 GPU Jupyter session before training.')


Stage VoxCeleb under `$PROJECT/caarma-data` before continuing. This notebook deliberately does not place Kaggle credentials in the repository or download data from a compute job.

In [ ]:
for required_path in (TRAIN_ROOT, TEST_ROOT, TRIAL_PATH, META_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required input: {required_path}')
print('Training root:', TRAIN_ROOT)
print('Evaluation root:', TEST_ROOT)
print('Metadata:', META_PATH)


In [ ]:
if TRAIN_CSV.exists():
    train_df = pd.read_csv(TRAIN_CSV)
    required_columns = {'utt_spk_int_labels', 'utt_paths'}
    if not required_columns.issubset(train_df.columns):
        raise ValueError(f'Existing training CSV is missing {required_columns - set(train_df.columns)}')
    print('Reusing existing training CSV without overwriting it:', TRAIN_CSV)
else:
    speakers = sorted(path.name for path in TRAIN_ROOT.iterdir() if path.is_dir())
    rows = [
        {'utt_spk_int_labels': label, 'utt_spk_id': speaker, 'utt_paths': str(wav_path)}
        for label, speaker in enumerate(speakers)
        for wav_path in sorted((TRAIN_ROOT / speaker).rglob('*.wav'))
    ]
    train_df = pd.DataFrame(rows)
    if train_df.empty:
        raise RuntimeError(f'No WAV files found below {TRAIN_ROOT}')
    TRAIN_CSV.parent.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(TRAIN_CSV, index=False)
num_speakers = int(train_df['utt_spk_int_labels'].nunique())
print(f'Wrote {len(train_df):,} utterances from {num_speakers} speakers to {TRAIN_CSV}')


In [ ]:
from collections import Counter
from functions.nationality import build_label_nationality_map, eligible_peer_count

label_to_nationality = build_label_nationality_map(TRAIN_CSV, META_PATH)
counts = Counter(label_to_nationality.values())
print(f'Matched metadata: {len(label_to_nationality)}/{num_speakers} speakers')
print(f'Nationalities: {len(counts)}; labels with a distinct eligible peer: {eligible_peer_count(label_to_nationality)}')
print('Largest groups:', counts.most_common(10))
if not label_to_nationality:
    raise ValueError('No train labels matched the VoxCeleb metadata.')
if eligible_peer_count(label_to_nationality) == 0:
    raise ValueError('No same-nationality speaker pairs exist.')


In [ ]:
SAVE_DIR.mkdir(parents=True, exist_ok=True)
config = {
    'model': 'MFA-CONFORMER', 'features': 'FBank_New',
    'augmentations': {'add_noise': False, 'add_reverb': False, 'drop_freq': False, 'drop_chunk': False},
    'dataset': str(TRAIN_CSV), 'USE_WANDB': False, 'wandb_project': 'mixup',
    'criterion': 'AMSoftmaxGAN', 'init_lr': 0.001, 'epochs': 30, 'weight_decay': 1e-7,
    'trial_path': str(TRIAL_PATH), 'root': str(TEST_ROOT) + os.sep, 'warmup_step': 2000,
    'batch_size': 50, 'num_workers': NUM_WORKERS, 'second': 3, 'do_augmentation': False,
    'num_spk': num_speakers, 'mixup': True, 'nationality_mixup': True,
    'vox1_meta_path': str(META_PATH), 'embedding_dim': 192, 'checkpoint_path': None,
    'save_dir': str(SAVE_DIR), 'title': 'caarma_nationality',
    'score_output_prefix': str(SAVE_DIR / 'caarma_nationality'), 'devices': 1,
}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print(CONFIG_PATH.read_text())


In [ ]:
trials = pd.read_csv(TRIAL_PATH, sep=r'\s+', header=None)
example_wav = TEST_ROOT / str(trials.iloc[0, 1])
print('Example evaluation WAV:', example_wav)
if not example_wav.exists():
    raise FileNotFoundError('TEST_ROOT does not match trial-relative paths.')


In [ ]:
command = [sys.executable, '-u', 'mixup_nationality.py', '--config', str(CONFIG_PATH), '--mode', 'train', '--nationality-mixup']
print('Running:', ' '.join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)


For a detached 48-hour run, close the notebook and submit `sbatch -A <allocation> bridges2/train_nationality.sbatch`. Override `CAARMA_REPO_ROOT`, `CAARMA_CONFIG`, or `AI_MODULE` as needed.